<a href="https://colab.research.google.com/github/SesSic/IAInvestigacionCurso/blob/main/sesion-03-limpieza-pandas/sesion03_completo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 3 — Gestión, limpieza y preparación de datos

**Temas:** importación de CSV/Excel, valores faltantes, transformación, validación y organización de datos

**Duración:** 3 horas

## Objetivo de la sesión
- Detectar valores faltantes, duplicados e inconsistencias en una base real
- Corregir texto mal escrito (mayúsculas, espacios) y tipos de dato incorrectos
- Eliminar duplicados y validar que la base quede lista para análisis
- Exportar una base limpia, lista para usar en las próximas sesiones

> 💡 Esto reemplaza lo que en Excel harías revisando fila por fila, o con "Buscar y reemplazar" — aquí se hace en unas pocas líneas, y es repetible cada vez que te llegue una base nueva con los mismos problemas.

---## 🟦 Bloque 1 (50 min) — Detectar los problemas de una base real

### 1.1 Cargar la base "sucia"

Vamos a trabajar con una versión de la base de notas que tiene los errores típicos que encontrarás en tus propios datos: valores vacíos, texto donde debería ir un número, mayúsculas inconsistentes, filas repetidas.

In [5]:
import pandas as pd
url = "https://raw.githubusercontent.com/SesSic/IAInvestigacionCurso/main/datasets/ejemplo_notas_sucio.csv"
datos = pd.read_csv(url)
datos

,estudiante,facultad,grupo,nota_parcial1,nota_parcial2,asistencia_pct
0,Ana Torres,Ciencias Sociales,A,7.5,8.2,95.0
1,Luis Paredes,ingenieria,A,6.8,7.0,80.0
2,Maria Chango,Ciencias Sociales,B,NaN,9.1,100.0
3,Carlos Vega,Educación,B,5.5,6.2,70.0
4,Sofia Ramos,Ingeniería,a,9.0,9.4,98.0
5,Diego Moreno,Salud,B,7.2,7.8,NaN
6,Paula Ibarra,Educación,A,seis,6.5,75.0
7,Jorge Salazar,Salud,B,8.1,8.0,92.0
8,Elena Cueva,Ciencias Sociales,A,7.8,8.5,90.0
9,Andres Puma,INGENIERIA,B,6.5,6.9,82.0


### 1.2 Primer diagnóstico: ¿qué tan sucia está?

Antes de arreglar nada, hay que medir el problema.

In [6]:
datos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   estudiante      17 non-null     object 
 1   facultad        17 non-null     object 
 2   grupo           17 non-null     object 
 3   nota_parcial1   16 non-null     object 
 4   nota_parcial2   16 non-null     float64
 5   asistencia_pct  16 non-null     float64
dtypes: float64(2), object(4)
memory usage: 948.0+ bytes


### 1.3 Contar valores faltantes por columna

In [7]:
valores_faltantes = datos.isna().sum()
print(valores_faltantes)

estudiante        0
facultad          0
grupo             0
nota_parcial1     1
nota_parcial2     1
asistencia_pct    1
dtype: int64


### 1.4 Detectar filas duplicadas

In [8]:
print("Filas duplicadas:", datos.duplicated().sum())
datos[datos.duplicated()]

Filas duplicadas: 2


,estudiante,facultad,grupo,nota_parcial1,nota_parcial2,asistencia_pct
15,Camila Andrade,Educación,B,6.7,7.1,79.0
16,Jorge Salazar,Salud,B,8.1,8.0,92.0


---### 🧪 Ejercicio 1 — Diagnóstico de tu propia base

Si trajiste tu archivo, cárgalo y repite estos 3 chequeos (`.info()`, `.isna().sum()`, `.duplicated().sum()`). Si no, sigue con la base de ejemplo.

In [ ]:
# Carga tu propio archivo aquí si lo tienes (sube el archivo primero con el ícono de carpeta 📁)
mis_datos = pd.read_csv("tu_archivo.csv")
mis_datos.info()

---## ☕ Descanso — 15 minutos

---## 🟩 Bloque 2 (50 min) — Corregir texto y tipos de dato

### 2.1 Espacios y mayúsculas inconsistentes

Nota cómo `"Ingeniería"`, `"ingenieria"` e `"INGENIERIA"` deberían ser la misma categoría, pero para la computadora son 3 categorías distintas.

In [9]:
print(datos["facultad"].value_counts())

facultad
Ciencias Sociales    4
Educación            4
Salud                4
Ingeniería           2
ingenieria           1
Educación            1
INGENIERIA           1
Name: count, dtype: int64


**Corrección:** quitamos espacios extra con `.str.strip()` y normalizamos mayúsculas con `.str.title()`.

In [10]:
datos["facultad"] = datos["facultad"].str.strip().str.title()
print(datos["facultad"].value_counts())

facultad
Educación            5
Ciencias Sociales    4
Salud                4
Ingenieria           2
Ingeniería           2
Name: count, dtype: int64


### 2.2 Lo mismo para la columna 'grupo' (A/B en mayúscula siempre)

In [11]:
print(datos["grupo"].value_counts())
datos["grupo"] = datos["grupo"].str.strip().str.upper()
print(datos["grupo"].value_counts())

grupo
B    10
A     6
a     1
Name: count, dtype: int64
grupo
B    10
A     7
Name: count, dtype: int64


### 2.3 Texto donde debería ir un número

La fila de Paula Ibarra tiene `"seis"` en vez de `6` en `nota_parcial1`. `pd.to_numeric()` con `errors="coerce"` convierte lo que puede a número, y lo que no puede lo vuelve `NaN` (valor faltante) — así lo detectamos en vez de que rompa todo el análisis.

In [12]:
datos["nota_parcial1"] = pd.to_numeric(datos["nota_parcial1"], errors="coerce")
datos["nota_parcial1"]

,nota_parcial1
0,7.5
1,6.8
2,NaN
3,5.5
4,9.0
5,7.2
6,NaN
7,8.1
8,7.8
9,6.5


---### 🧪 Ejercicio 2 — Normaliza una columna de texto de tu base

Elige una columna categórica de tu propia base (o de esta) y aplica `.str.strip().str.title()` (o `.upper()` si corresponde) para normalizarla.

In [ ]:
# TODO: elige una columna de texto y normalízala
datos["columna"] = datos["columna"].str.strip().str.title()

---## ☕ Descanso — 15 minutos

---## 🟨 Bloque 3 (50 min) — Valores faltantes, duplicados y exportar la base limpia

### 3.1 Decidir qué hacer con los valores faltantes

Dos opciones típicas:

**eliminar** la fila, o **rellenar** con un valor razonable (ej. el promedio de la columna).

No hay una regla única — depende de cuántos datos faltan y qué tan crítica es esa columna.

In [13]:
# Opción A: eliminar filas donde falte 'nota_parcial1' o 'nota_parcial2'
datos_sin_nulos = datos.dropna(subset=["nota_parcial1", "nota_parcial2"])
print("Filas antes:", len(datos), "| Filas después:", len(datos_sin_nulos))

Filas antes: 17 | Filas después: 14


In [14]:
# Opción B: rellenar 'asistencia_pct' faltante con el promedio de esa columna
promedio_asistencia = datos["asistencia_pct"].mean()
datos["asistencia_pct"] = datos["asistencia_pct"].fillna(promedio_asistencia)
datos["asistencia_pct"]

,asistencia_pct
0,95.0000
1,80.0000
2,100.0000
3,70.0000
4,98.0000
5,86.3125
6,75.0000
7,92.0000
8,90.0000
9,82.0000


### 3.2 Eliminar filas duplicadas

In [15]:
datos_limpios = datos.drop_duplicates()
print("Filas antes:", len(datos), "| Filas después de quitar duplicados:", len(datos_limpios))

Filas antes: 17 | Filas después de quitar duplicados: 15


### 3.3 Validación final

Antes de dar por buena la limpieza, siempre repite el diagnóstico del Bloque 1 para confirmar que ya no hay problemas.

In [16]:
print(datos_limpios.isna().sum())
print("Duplicados restantes:", datos_limpios.duplicated().sum())

estudiante        0
facultad          0
grupo             0
nota_parcial1     2
nota_parcial2     1
asistencia_pct    0
dtype: int64
Duplicados restantes: 0


### 3.4 Exportar la base limpia

Guarda el resultado para no tener que repetir esta limpieza cada vez.

In [17]:
datos_limpios.to_csv("notas_limpio.csv", index=False)
print("Archivo guardado. Lo encuentras en el panel de archivos de la izquierda (ícono de carpeta 📁).")

Archivo guardado. Lo encuentras en el panel de archivos de la izquierda (ícono de carpeta 📁).


---### 🧪 Ejercicio final — Limpia tu propia base

Si trajiste tu archivo, aplícale el mismo proceso completo:
1. Diagnóstico (`.info()`, `.isna().sum()`, `.duplicated().sum()`)
2. Normalizar texto (`.str.strip().str.title()`)
3. Corregir tipos de dato si aplica (`pd.to_numeric(..., errors="coerce")`)
4. Decidir: `.dropna()` o `.fillna()` según el caso
5. `.drop_duplicates()`
6. Validar de nuevo7. `.to_csv()` para guardar la versión limpia

In [ ]:
# Aplica aquí los pasos anteriores a tu propia base

---## ✅ Cierre de la sesión

Hoy lograste:
- ✅ Diagnosticar una base de datos (valores faltantes, duplicados, tipos incorrectos)- ✅ Normalizar texto inconsistente
- ✅ Corregir columnas con tipo de dato incorrecto
- ✅ Eliminar duplicados y decidir qué hacer con los valores faltantes
- ✅ Exportar una base limpia lista para análisis

**Para la próxima sesión (Estadística):** vamos a usar esta misma base ya limpia para calcular estadísticas descriptivas y comparar grupos — trae tu propia base ya depurada si la tienes.

📎 Recuerda: `Archivo → Guardar una copia en Drive` para no perder tu trabajo.